# ¿Gana más de 50 000 USD? Correlación y ética

**Sección 3.12 del libro**

Datos del censo de EE. UU. de 1994 (dataset *Adult*, UCI). Usamos correlaciones para decidir qué columnas quitar y reflexionamos: **la IA aprende los sesgos que haya en los datos**.

> **Basado en:** Karatas, M. (2024). *Developing AI Applications: An Introduction*, cap. 3 «Artificial Neural Networks». Rheinwerk Publishing. Código original del autor, adaptado a Google Colab y comentado en español con fines educativos.


In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split

## Cargar los datos

Antes de ejecutar esta celda, sube **adult-2.csv** al panel de archivos de Colab (ícono de carpeta → botón de subir). Si no lo tienes a mano, usa la línea comentada: lee el archivo directamente desde la página de la presentación.

In [ ]:
path = "adult-2.csv"
# path = "https://sinergia-iesa.github.io/introduccion-a-la-creacion-de-redes-neuronales/data/adult-2.csv"   # alternativa sin subir el archivo

data = pd.read_csv(path, delimiter=';')
print(data.head())
print("Columnas con huecos:", data.columns[data.isnull().any()])

En `adult-2.csv` los huecos originales (`?`) ya se reemplazaron por la palabra `empty`, como hace el autor en una hoja de cálculo.

## Correlaciones

In [ ]:
# Correlaciones: ¿qué columnas aportan poco?
correlations = data.corr(numeric_only=True)
correlations_abs_sum = correlations.abs().sum()
print('Suma de correlaciones (en valor absoluto)')
print('-' * 30)
print(correlations_abs_sum)
print('\nLas 5 más débiles')
print('-' * 30)
print(correlations_abs_sum.nsmallest(5))

## Preparar columnas

- Quitar las de correlación más baja.
- Texto → números (`cat.codes`) o → OHE (`get_dummies`).

*El libro escribe `'occuaption'` (errata); en este archivo la columna se llama `occupation`.*

In [ ]:
data.drop(['fnlwgt', 'capital.loss', 'capital.gain'], axis=1, inplace=True)   # correlación baja

# Lo que queremos predecir: ¿gana más de 50 000 USD al año?
col = pd.get_dummies(data['income'], dtype=float)
data = data.drop(['income'], axis=1)

# Columnas de texto → números 0…n
conv_num = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'hours.per.week', 'native.country']
data[conv_num] = data[conv_num].astype('category')
data[conv_num] = data[conv_num].apply(lambda x: x.cat.codes)

# Columnas de texto → OHE
data = pd.get_dummies(data, columns=['race', 'sex'], dtype=float)
print(data.shape)

## Construir, entrenar y probar

In [ ]:
train_data, test_data, train_col, test_col = train_test_split(data, col, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(data.shape[1],)),   # tantas entradas como columnas
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=10)

test_loss, test_acc = model.evaluate(test_data, test_col)
print('Test accuracy:', test_acc)

## ⚠️ Para pensar

Si en los datos ciertos grupos ganan menos por discriminación, la red **aprende y repite** ese patrón. Correlación no es causalidad. En el siguiente cuaderno se eliminan `race`, `sex` y `native.country`.